# 내부 CLIP 유사도 시세 코퍼스 검증 (CH020)

카테고리 파인튜닝 백본(last-N=4)으로 AI-Hub 이미지 코퍼스를 만들고, CLIP 유사도 검색이
그럴듯한 카테고리를 top-K로 찾아내는지 확인한다. **코퍼스는 전부 합성 데이터**(AI-Hub
이미지 + 가상 가격)라 가격 숫자 자체는 무의미하다 — 메커니즘이 도는지만 본다.

In [ ]:
# --- 환경 설정 ---
import subprocess
import sys
import time
from pathlib import Path


def run(script, *args):
    cmd = [sys.executable, str(HERE / script)] + [str(a) for a in args]
    print("$", " ".join(cmd))
    start = time.time()
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        bufsize=1,
    )
    for line in process.stdout:
        elapsed = time.time() - start
        print("[%7.1fs]" % elapsed, line, end="")
    process.wait()
    elapsed = time.time() - start
    if process.returncode == 0:
        print("")
        print("성공 - 총 %.1f초" % elapsed)
    else:
        print("")
        print("실패 returncode=%s - 총 %.1f초" % (process.returncode, elapsed))
    return process


HERE = Path("/home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune")
assert (HERE / "aihub_dataset.py").exists()
print("good")

## 1. 코퍼스 시딩

`--image-root`만 실제 원천데이터(이미지) 루트로 바꿀 것. `--label-root`는 이전에
`aihub_dataset.py scan/index`에서 쓰던 clothes 라벨 루트와 같다.

In [ ]:
run(
    "aihub_seed_pricing_corpus.py",
    "--label-root", "/home/j-i15a108/work/aihub/extracted/clothes",
    "--image-root", "<원천데이터 이미지 루트>",
    "--checkpoint-path", HERE / "artifacts_finetune" / "vit_last4_category_masked_lr1e-05" / "checkpoint.pt",
    "--out-dir", HERE.parent.parent.parent / "app" / "pricing" / "data",
    "--max-per-category", 200,
    "--limit", 2000,
)

## 2. 임베딩 + 유사도 검색 스모크 테스트

네트워크 없이, 저장된 코퍼스에서 top-5 유사 항목을 바로 확인한다.

In [ ]:
AI_ROOT = HERE.parent.parent.parent
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

from PIL import Image
from app.pricing import clip_embed, internal_corpus

checkpoint_path = str(HERE / "artifacts_finetune" / "vit_last4_category_masked_lr1e-05" / "checkpoint.pt")
corpus_dir = str(AI_ROOT / "app" / "pricing" / "data")

image = Image.open(HERE.parent / "product-info-check" / "inputs" / "apc.png").convert("RGB")
vec = clip_embed.embed_image(image, checkpoint_path=checkpoint_path, model_id="patrickjohncyh/fashion-clip")
corpus = internal_corpus.get_corpus(corpus_dir)

for item, score in internal_corpus.query_similar(vec, corpus, top_k=5, min_similarity=0.55):
    print(f"{item.aihub_category:12s} price={item.synthetic_price:>8,} sim={score:.3f}")